In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col



data = [[0, 0, 'start', 0.712], [0, 0, 'end', 1.52], [0, 1, 'start', 3.14], [0, 1, 'end', 4.12], [1, 0, 'start', 0.55], [1, 0, 'end', 1.55], [1, 1, 'start', 0.43], [1, 1, 'end', 1.42], [2, 0, 'start', 4.1], [2, 0, 'end', 4.512], [2, 1, 'start', 2.5], [2, 1, 'end', 5]]
activity = pd.DataFrame(data, columns=['machine_id', 'process_id', 'activity_type', 'timestamp']). \
    astype({'machine_id':'Int64', 'process_id':'Int64', 'activity_type':'object', 'timestamp':'Float64'})


spark = SparkSession.builder.appName("ActivityData").getOrCreate()

activity_df = spark.createDataFrame(activity)

activity_df.show()

activity_df.printSchema()

activity_join_df = activity_df.alias('a1')\
    .join(activity_df.alias('a2'), on=[(col('a1.machine_id') == col('a2.machine_id')) & (col('a1.process_id') == col('a2.process_id')) & (col('a1.activity_type') == "start") & (col('a2.activity_type') == "end") ], how='inner')\
    .select(col('a1.machine_id'), col('a1.process_id'), col('a1.timestamp'), col('a2.timestamp'), (col('a2.timestamp') - col('a1.timestamp')).alias('duration'),)

activity_join_df.groupBy('machine_id').agg({'duration':'avg'}).withColumnRenamed('avg(duration)', 'avg_duration').select('machine_id', 'avg_duration').show()

spark.stop()


+----------+----------+-------------+---------+
|machine_id|process_id|activity_type|timestamp|
+----------+----------+-------------+---------+
|         0|         0|        start|    0.712|
|         0|         0|          end|     1.52|
|         0|         1|        start|     3.14|
|         0|         1|          end|     4.12|
|         1|         0|        start|     0.55|
|         1|         0|          end|     1.55|
|         1|         1|        start|     0.43|
|         1|         1|          end|     1.42|
|         2|         0|        start|      4.1|
|         2|         0|          end|    4.512|
|         2|         1|        start|      2.5|
|         2|         1|          end|      5.0|
+----------+----------+-------------+---------+

root
 |-- machine_id: long (nullable = true)
 |-- process_id: long (nullable = true)
 |-- activity_type: string (nullable = true)
 |-- timestamp: double (nullable = true)

+----------+------------+
|machine_id|avg_duration|
+------